## Imports

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('/home/ec2-user/multimodal-rag'))

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import json
import numpy as np
import torch
from tqdm.notebook import tqdm
from colpali_engine.models import ColQwen3_5, ColQwen3_5Processor
from scripts.vlm import VLM
from scripts.evaluation import evaluate_dataframe

## Configurations

In [3]:
SEED = 19
QUESTION_PATHS = {
    "FinancialReport" : "../data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl",
    "TechReport" : "../data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl",
    "TechSlides" : "../data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"
}
RESULTS_PATH = "./results/image_rag"
model_id = "Qwen/Qwen3-VL-8B-Instruct"
checkpoint_path = "../Qwen3-VL-8B-Instruct"
COLQWEN_MODEL = "athrael-soju/colqwen3.5-4.5B-v3"
VECTORS_DIR = "../scripts/embeddings_numpy"
TOP_K = 3
RETRIEVAL_FILES = {
    "FinancialReport": f"{RESULTS_PATH}/retrieval_finreport.json",
    "TechReport":      f"{RESULTS_PATH}/retrieval_techreport.json",
    "TechSlides":      f"{RESULTS_PATH}/retrieval_techslides.json",
}

## Load Data

In [7]:
dataframes = {}

for key, path in QUESTION_PATHS.items():
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "example_index": obj.get("example_index"),
                "question": obj.get("question"),
                "answer": obj.get("answer"),
                "intended_img": obj.get("image_path")
            })
    dataframes[key] = pd.DataFrame(records).set_index("example_index")

splits = {}
for key, df in dataframes.items():
    train = df.sample(frac=0.8, random_state=SEED)
    test = df.drop(train.index)
    splits[key] = {"train": train, "test": test}
    print(f"{key}: {len(train)} train / {len(test)} test")

FinancialReport: 682 train / 171 test
TechReport: 1035 train / 259 test
TechSlides: 1083 train / 271 test


## Retrieval Phase

Run the cells in this section to retrieve top-k page images per query using ColQwen and save image paths to JSON. Then **restart the kernel** before running the VLM Inference section — ColQwen and QwenVL-8B cannot coexist in GPU memory.

In [4]:
colqwen_processor = ColQwen3_5Processor.from_pretrained(COLQWEN_MODEL)
colqwen = ColQwen3_5.from_pretrained(
    COLQWEN_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="sdpa",
)
colqwen.eval()

# Load all pre-computed page vectors onto GPU (~1.3 GB in bfloat16)
page_vectors = torch.from_numpy(
    np.load(f"{VECTORS_DIR}/vectors.npy")
).to(torch.bfloat16).cuda()

with open(f"{VECTORS_DIR}/metadata.json") as f:
    page_meta = json.load(f)

print(f"Loaded {len(page_meta)} page embeddings: {page_vectors.shape}")

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/725 [00:00<?, ?it/s]

Loaded 3631 page embeddings: torch.Size([3631, 767, 320])


Retrieve top-k images for every test query and save image paths to JSON.

In [5]:
def search(query: str, top_k: int = TOP_K) -> list:
    with torch.no_grad():
        batch = colqwen_processor.process_queries([query]).to("cuda")
        q_emb = colqwen(**batch)[0].to(torch.bfloat16)          # (q_patches, 320)
    # MaxSim: for each page, sum over query patches of (max over page patches of dot product)
    scores = torch.einsum("qd,npd->qnp", q_emb, page_vectors).max(dim=2).values.sum(dim=0)
    top_idx = scores.topk(top_k * 5).indices.tolist()
    paths = [page_meta[i]["path"] for i in top_idx if os.path.exists(page_meta[i]["path"])]
    return paths[:top_k]

os.makedirs(RESULTS_PATH, exist_ok=True)
for key, split in splits.items():
    test_df = split['test']
    retrieval_path = RETRIEVAL_FILES[key]

    print(f"Retrieving for {key}...")
    records = []
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        records.append({
            "example_index": idx,
            "dataset": key,
            "question": row['question'],
            "reference": row['answer'],
            "retrieved_images": search(row['question']),
        })

    with open(retrieval_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"  Saved {len(records)} records → {retrieval_path}")

print("\n✅ Retrieval complete. Restart the kernel before running VLM Inference.")

Retrieving for FinancialReport...


  0%|          | 0/171 [00:00<?, ?it/s]

  Saved 171 records → ./results/image_rag/retrieval_finreport.json
Retrieving for TechReport...


  0%|          | 0/259 [00:00<?, ?it/s]

  Saved 259 records → ./results/image_rag/retrieval_techreport.json
Retrieving for TechSlides...


  0%|          | 0/271 [00:00<?, ?it/s]

  Saved 271 records → ./results/image_rag/retrieval_techslides.json

✅ Retrieval complete. Restart the kernel before running VLM Inference.


## VLM Inference

After restarting the kernel, re-run Imports, Configurations, and Load Data, then run the cells below.

In [3]:
retrieval_lookup = {}
for key, path in RETRIEVAL_FILES.items():
    with open(path) as f:
        for rec in json.load(f):
            retrieval_lookup[(key, rec["example_index"])] = rec["retrieved_images"]
print(f"Loaded {len(retrieval_lookup)} retrieval records.")

Loaded 701 retrieval records.


In [4]:
vlm = VLM(model_id, checkpoint_path)

--- Local model found at: ../Qwen3-VL-8B-Instruct ---
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Multi-thread loading shards: 100% Completed | 4/4 [02:13<00:00, 33.40s/it]
Capturing batches (bs=24 avail_mem=1.75 GB):   0%|          | 0/7 [00:00<?, ?it/s]2026-04-24 16:39:56,997 - CUTE_DSL - WARNING - [handle_import_error] - Unexpected error during package walk: cutlass.cute.experimental
[2026-04-24 16:39:56] Unexpected error during package walk: cutlass.cute.experimental
Capturing batches (bs=1 avail_mem=1.68 GB): 100%|██████████| 7/7 [00:05<00:00,  1.34it/s] 


In [8]:
os.makedirs(RESULTS_PATH, exist_ok=True)
output_filepath = f"{RESULTS_PATH}/image_rag_results.jsonl"

with open(output_filepath, "w", encoding="utf-8") as f:
    for key, split in splits.items():
        test_df = split['test']
        print(f"Running Inference for {key}")

        for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
            retrieved_images = retrieval_lookup.get((key, idx))
            if not retrieved_images:
                print(f"  WARNING: no images for {key} example {idx}, skipping")
                continue

            content = [
                {"type": "image", "image": img_path, "max_pixels": 1120390}
                for img_path in retrieved_images
                if os.path.exists(img_path)
            ]
            content.append({"type": "text", "text": row['question']})

            messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": "You are a question answering assistant for corporate applications. Respond in one sentence using all available information."}]
                },
                {"role": "user", "content": content}
            ]

            try:
                response = vlm.generate(messages)
                generated = response['text']
            except Exception as e:
                print(f"  Error on example {idx}: {e}")
                continue

            result_obj = {
                "dataset": key,
                "example_index": idx,
                "question": row['question'],
                "generated_answer": generated,
                "ground_truth": row['answer'],
                "intended_img": row['intended_img']
            }

            f.write(json.dumps(result_obj) + "\n")
            f.flush()

print(f"✅ Inference complete. Results saved to {output_filepath}")

Running Inference for FinancialReport


  0%|          | 0/171 [00:00<?, ?it/s]

Running Inference for TechReport


  0%|          | 0/259 [00:00<?, ?it/s]

Running Inference for TechSlides


  0%|          | 0/271 [00:00<?, ?it/s]

✅ Inference complete. Results saved to ./results/image_rag/image_rag_results.jsonl


## Evaluation

In [4]:
input_filepath = f"{RESULTS_PATH}/image_rag_results.jsonl"
report_filepath = f"{RESULTS_PATH}/image_rag_evaluation_report.txt"
detailed_jsonl_filepath = f"{RESULTS_PATH}/image_rag_evaluated_details.jsonl"

print(f"Loading data from {input_filepath}...")
records = []
if os.path.exists(input_filepath):
    with open(input_filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
else:
    raise FileNotFoundError(f"Could not find {input_filepath}. Check your paths!")

df_results = pd.DataFrame(records)
print(f"Loaded {len(df_results)} records across datasets: {df_results['dataset'].unique()}")
print("\nStarting evaluation (this may take a few minutes depending on BERTScore)...")
evaluated_df = evaluate_dataframe(df_results, report_filepath)

evaluated_df.to_json(detailed_jsonl_filepath, orient="records", lines=True)
print(f"✅ Detailed row-by-row scores saved to {detailed_jsonl_filepath}")

Loading data from ./results/image_rag/image_rag_results.jsonl...
Loaded 701 records across datasets: ['FinancialReport' 'TechReport' 'TechSlides']

Starting evaluation (this may take a few minutes depending on BERTScore)...
Calculating BLEU scores...
Calculating ROUGE scores...
Calculating BERT scores (this may take a moment)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Report successfully saved to ./results/image_rag/image_rag_evaluation_report.txt
✅ Detailed row-by-row scores saved to ./results/image_rag/image_rag_evaluated_details.jsonl
